<a href="https://colab.research.google.com/github/kenzabelk/M2-THESIS/blob/main/M2_THESIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# BLOC 1 — Chargement depuis Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df_real = pd.read_csv('/content/drive/MyDrive/True.csv')
df_fake = pd.read_csv('/content/drive/MyDrive/Fake.csv')

df_real["label"] = 0
df_fake["label"] = 1

df = pd.concat([df_real, df_fake], ignore_index=True)

print("Taille du dataset :", df.shape)
print("\nDistribution des classes :")
print(df["label"].value_counts())

Mounted at /content/drive
Taille du dataset : (44898, 5)

Distribution des classes :
label
1    23481
0    21417
Name: count, dtype: int64


In [2]:
# ============================================================
# BLOC 2 — Nettoyage du texte (Text Preprocessing)
# Description : On nettoie le texte brut pour supprimer
# le bruit inutile : URLs, ponctuation, chiffres, espaces
# en trop, et on met tout en minuscules.
# On combine le titre + le texte en une seule colonne "content"
# ============================================================

import re

def clean_text(text):
    text = str(text).lower()                        # minuscules
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # supprimer URLs
    text = re.sub(r'\[.*?\]', '', text)             # supprimer [...]
    text = re.sub(r'[^a-z\s]', '', text)            # garder lettres seulement
    text = re.sub(r'\s+', ' ', text).strip()        # espaces multiples
    return text

# Combiner titre + texte
df["content"] = df["title"] + " " + df["text"]

# Appliquer le nettoyage
df["content_clean"] = df["content"].apply(clean_text)

# Vérification
print("Exemple AVANT nettoyage :")
print(df["content"][0][:200])
print("\nExemple APRÈS nettoyage :")
print(df["content_clean"][0][:200])


Exemple AVANT nettoyage :
As U.S. budget fight looms, Republicans flip their fiscal script WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of

Exemple APRÈS nettoyage :
as us budget fight looms republicans flip their fiscal script washington reuters the head of a conservative republican faction in the us congress who voted this month for a huge expansion of the natio


In [3]:
# ============================================================
# BLOC 3 — Division du dataset (Train / Validation / Test)
# Description : On divise le dataset en 3 parties :
# - 70% pour entraîner les modèles (training)
# - 15% pour ajuster les paramètres (validation)
# - 15% pour évaluer les performances finales (test)
# ============================================================

from sklearn.model_selection import train_test_split

X = df["content_clean"]
y = df["label"]

# Première division : 70% train, 30% reste
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Deuxième division : 50% val, 50% test (soit 15%/15% du total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Vérification
print("Taille Train      :", len(X_train), "articles")
print("Taille Validation :", len(X_val),   "articles")
print("Taille Test       :", len(X_test),  "articles")
print("\nTotal :", len(X_train) + len(X_val) + len(X_test), "articles")

Taille Train      : 31428 articles
Taille Validation : 6735 articles
Taille Test       : 6735 articles

Total : 44898 articles


In [4]:
# ============================================================
# BLOC 4 — Vectorisation TF-IDF
# Description : Les modèles ML ne comprennent pas les mots —
# il faut les convertir en chiffres. On utilise TF-IDF
# (Term Frequency - Inverse Document Frequency) qui donne
# un score d'importance à chaque mot dans le texte.
# On garde les 50,000 mots les plus fréquents.
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# Création du vectoriseur
tfidf = TfidfVectorizer(max_features=50000, stop_words="english")

# Apprentissage sur le train UNIQUEMENT, puis transformation
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

# Vérification
print("Forme de X_train_tfidf :", X_train_tfidf.shape)
print("Forme de X_val_tfidf   :", X_val_tfidf.shape)
print("Forme de X_test_tfidf  :", X_test_tfidf.shape)
print("\nVectorisation TF-IDF terminée ✓")

Forme de X_train_tfidf : (31428, 50000)
Forme de X_val_tfidf   : (6735, 50000)
Forme de X_test_tfidf  : (6735, 50000)

Vectorisation TF-IDF terminée ✓


In [5]:
# ============================================================
# BLOC 5 — Entraînement des modèles ML classiques
# Description : On entraîne 3 modèles classiques sur les
# données TF-IDF et on compare leurs performances :
# - Naive Bayes (NB) : modèle probabiliste simple et rapide
# - Logistic Regression (LR) : modèle linéaire de référence
# - Random Forest (RF) : ensemble d'arbres de décision
# Chaque modèle est évalué sur le set de validation.
# ============================================================

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Dictionnaire des modèles
models = {
    "Naive Bayes"        : MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest"      : RandomForestClassifier(n_estimators=100, random_state=42),
}

# Entraînement et évaluation sur validation
results = {}
for name, model in models.items():
    print(f"Entraînement de {name}...")
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_val_tfidf)
    results[name] = {
        "Accuracy" : round(accuracy_score(y_val, y_pred) * 100, 2),
        "Precision": round(precision_score(y_val, y_pred) * 100, 2),
        "Recall"   : round(recall_score(y_val, y_pred) * 100, 2),
        "F1-score" : round(f1_score(y_val, y_pred) * 100, 2),
    }
    print(f"  ✓ Accuracy: {results[name]['Accuracy']}% | F1: {results[name]['F1-score']}%")

print("\nEntraînement terminé ✓")

Entraînement de Naive Bayes...
  ✓ Accuracy: 94.42% | F1: 94.67%
Entraînement de Logistic Regression...
  ✓ Accuracy: 98.6% | F1: 98.66%
Entraînement de Random Forest...
  ✓ Accuracy: 99.5% | F1: 99.52%

Entraînement terminé ✓


In [6]:
# ============================================================
# BLOC 6 — Modèle DistilBERT (Transformer)
# Description : DistilBERT est une version légère de BERT.
# Contrairement à TF-IDF, il comprend le contexte des mots.
# On l'entraîne sur nos données pour la classification
# binaire : fake (1) vs real (0).
# ============================================================

# Installation des librairies nécessaires
!pip install transformers torch -q

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch

# Vérification GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil utilisé :", device)

# Chargement du tokenizer DistilBERT
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
print("Tokenizer chargé ✓")

# On prend un sous-ensemble pour aller plus vite
# (5000 train, 1000 val) — suffisant pour démontrer le pipeline
X_train_bert = X_train.values[:5000]
y_train_bert = y_train.values[:5000]
X_val_bert   = X_val.values[:1000]
y_val_bert   = y_val.values[:1000]

# Tokenisation
print("Tokenisation en cours...")
train_enc = tokenizer(list(X_train_bert), truncation=True, padding=True,
                      max_length=256, return_tensors="pt")
val_enc   = tokenizer(list(X_val_bert),   truncation=True, padding=True,
                      max_length=256, return_tensors="pt")
print("Tokenisation terminée ✓")

Appareil utilisé : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer chargé ✓
Tokenisation en cours...
Tokenisation terminée ✓


In [7]:
# ============================================================
# BLOC 6 (suite) — Création du Dataset et entraînement
# Description : On crée un objet Dataset PyTorch, puis on
# entraîne DistilBERT pendant 2 epochs sur nos données.
# 1 epoch = le modèle voit tous les exemples d'entraînement
# une fois.
# ============================================================

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# Création du Dataset PyTorch
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = FakeNewsDataset(train_enc, y_train_bert)
val_dataset   = FakeNewsDataset(val_enc,   y_val_bert)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)

# Chargement du modèle DistilBERT
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2)
model.to(device)

# Optimiseur
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Entraînement — 2 epochs
print("Entraînement DistilBERT en cours...")
print("(Patience — environ 10-15 minutes sur CPU)\n")

for epoch in range(2):
    model.train()
    total_loss = 0
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if i % 50 == 0:
            print(f"  Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    print(f"\n✓ Epoch {epoch+1} terminée — Loss moyenne: {total_loss/len(train_loader):.4f}\n")

print("Entraînement terminé ✓")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Entraînement DistilBERT en cours...
(Patience — environ 10-15 minutes sur CPU)

  Epoch 1 | Batch 0/313 | Loss: 0.7175
  Epoch 1 | Batch 50/313 | Loss: 0.0297
  Epoch 1 | Batch 100/313 | Loss: 0.3637
  Epoch 1 | Batch 150/313 | Loss: 0.0072
  Epoch 1 | Batch 200/313 | Loss: 0.0039
  Epoch 1 | Batch 250/313 | Loss: 0.0057
  Epoch 1 | Batch 300/313 | Loss: 0.0020

✓ Epoch 1 terminée — Loss moyenne: 0.0683

  Epoch 2 | Batch 0/313 | Loss: 0.0019
  Epoch 2 | Batch 50/313 | Loss: 0.0012
  Epoch 2 | Batch 100/313 | Loss: 0.0061
  Epoch 2 | Batch 150/313 | Loss: 0.0017
  Epoch 2 | Batch 200/313 | Loss: 0.0009
  Epoch 2 | Batch 250/313 | Loss: 0.0007
  Epoch 2 | Batch 300/313 | Loss: 0.0085

✓ Epoch 2 terminée — Loss moyenne: 0.0049

Entraînement terminé ✓


In [8]:
# ============================================================
# BLOC 6 (suite) — Évaluation de DistilBERT sur le test set
# Description : On évalue le modèle entraîné sur les données
# de test et on affiche les métriques finales.
# ============================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch

model.eval()
all_preds = []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

y_pred_bert = all_preds
y_true_bert = y_val_bert

print("=== Résultats DistilBERT ===")
print(f"Accuracy  : {round(accuracy_score(y_true_bert, y_pred_bert)*100, 2)}%")
print(f"Precision : {round(precision_score(y_true_bert, y_pred_bert)*100, 2)}%")
print(f"Recall    : {round(recall_score(y_true_bert, y_pred_bert)*100, 2)}%")
print(f"F1-score  : {round(f1_score(y_true_bert, y_pred_bert)*100, 2)}%")

=== Résultats DistilBERT ===
Accuracy  : 99.6%
Precision : 100.0%
Recall    : 99.27%
F1-score  : 99.64%


In [9]:
# ============================================================
# BLOC 7 — Tableau comparatif final (tous les modèles)
# Description : On affiche le tableau complet avec toutes
# les métriques pour les 4 modèles sur le set de validation.
# ============================================================

print(f"{'Modèle':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-score':>10}")
print("-" * 65)
for name, metrics in results.items():
    print(f"{name:<25} {metrics['Accuracy']:>9}% {metrics['Precision']:>9}% {metrics['Recall']:>9}% {metrics['F1-score']:>9}%")

# Ajouter DistilBERT
distilbert_metrics = {
    'Accuracy' : round(accuracy_score(y_true_bert, y_pred_bert)*100, 2),
    'Precision': round(precision_score(y_true_bert, y_pred_bert)*100, 2),
    'Recall'   : round(recall_score(y_true_bert, y_pred_bert)*100, 2),
    'F1-score' : round(f1_score(y_true_bert, y_pred_bert)*100, 2),
}
print(f"{'DistilBERT':<25} {distilbert_metrics['Accuracy']:>9}% {distilbert_metrics['Precision']:>9}% {distilbert_metrics['Recall']:>9}% {distilbert_metrics['F1-score']:>9}%")

Modèle                      Accuracy  Precision     Recall   F1-score
-----------------------------------------------------------------
Naive Bayes                   94.42%     94.46%     94.89%     94.67%
Logistic Regression            98.6%     98.83%      98.5%     98.66%
Random Forest                  99.5%      99.6%     99.43%     99.52%
DistilBERT                     99.6%     100.0%     99.27%     99.64%
